# Construct bilateral US–China GPR (complete notebook)

**Supervisor:** develop *your* geopolitical measurement model (Bondarenko et al. 2026 SSRN spirit), not off-the-shelf GPR/Goldstein.

**Sample:** 1990-01 → 2022-02 (Saadaoui replication window, 386 months).

---
## What failed in the first run (read this)

If your plot was **empty 1990–2013** and data only from **2013-09** onward, the pipeline used **daily-only** GDELT URLs:

| Period | Correct file | Daily URL `YYYYMMDD.export.CSV.zip` |
|--------|----------------|-------------------------------------|
| 1990–2005 | `YYYY.zip` (yearly) | **404** |
| 2006–2013-03 | `YYYYMM.zip` (monthly) | **404** |
| 2013-04–2022 | daily zip | **200** |

So a “pilot = 1st day of each month” on **daily files only** skips 284 months. That is a **data URL mistake**, not “no US–China events before 2013.”

**Fix (this notebook):** yearly → monthly → daily archives per GDELT documentation.


## 1 — Research design: what we build vs what we do not

### Bondarenko et al. (euro-area GPR)
- Local newspapers → share of articles with **war/terror/conflict** keywords each month.
- Country indices aggregated to a regional GPR.

### Our US–China analog (feasible without Factiva)
- **Numerator:** US↔China GDELT events where **source URL + actor text** match Caldara-style **threat** keywords.
- **Denominator:** all US↔China GDELT events that month.
- **Index:** `gpr_kw_share_t = n_kw / n_total`.
- **Shock:** `d2gpr_kw` = second difference (parallel to Saadaoui’s Δ²PRI).

### What we explicitly do **not** use as *your* index
- Caldara–Iacoviello `gpr` (validation only).
- `gdelt_goldstein_mean`, `gdel_events_monthly_clean.csv` aggregates without raw re-processing.
- FinBERT / PCA composites from prior notebooks.

### Limitation (state in thesis)
We keyword-match **event-linked text** (URLs, actor names), not full newspaper article bodies. Closer to Bondarenko than using downloaded scores; weaker than licensed article archives.


## 2 — Configuration


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')
NB_DIR = Path.cwd().resolve()
if str(NB_DIR) not in sys.path: sys.path.insert(0, str(NB_DIR))
import measurement_lib as ml

PATHS = ml.resolve_paths(NB_DIR)
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)

# Download scope:
#   'smoke'     = 1990 + 2010 yearly, one month, few days (~2 min)
#   'saadaoui'  = full window; 2013-04+ uses 1st-of-month daily (fast, ~1-2 h)
#   'saadaoui_dense' = full window + EVERY day 2013-04..2022 (overnight, GB+) 
DOWNLOAD_SCOPE = 'smoke'   # <-- change to 'saadaoui' for thesis sample
RUN_IV = True
H_MAX = 12

ml.save_keywords(PATHS)
EN_PAT, CN_PAT = ml.compile_keyword_patterns()
plan = ml.gdelt_download_plan(DOWNLOAD_SCOPE)
print(f'Download plan: {len(plan)} files | scope={DOWNLOAD_SCOPE}')
print('  years:', sum(1 for k,_ in plan if k=='year'))
print('  months:', sum(1 for k,_ in plan if k=='month'))
print('  days:', sum(1 for k,_ in plan if k=='day'))


## 3 — Keyword lists (measurement rules you own)

Threat keywords adapted from Caldara–Iacoviello. **Excluded:** country names (`china`, `beijing`) because the corpus is already US↔CHN — including them marks ~100% of rows.

Early GDELT rows (≤57 columns) often lack URLs; matching uses actor names + event codes in those months.


In [ ]:
print('EN threat keywords (%d):' % len(ml.GPR_KEYWORDS_EN))
print(', '.join(ml.GPR_KEYWORDS_EN[:12]), '...')
print('CN keywords:', ml.GPR_KEYWORDS_CN)
print('Saved:', PATHS['keywords'] / 'gpr_keyword_config.json')


## 4 — Download & label raw GDELT (resumable caches)

Caches:
- `raw/gdelt_yearly/YYYY.parquet`
- `raw/gdelt_monthly/YYYYMM.parquet`
- `raw/gdelt_daily/YYYYMMDD.parquet`

Re-running skips existing files. **After changing scope**, delete stale `gdelt_daily` files if you previously ran daily-only pilot.


In [ ]:
corpus = ml.build_corpus(PATHS, scope=DOWNLOAD_SCOPE, en_pat=EN_PAT, cn_pat=CN_PAT)
if corpus.empty:
    raise RuntimeError('No events — try DOWNLOAD_SCOPE="smoke" first, check network.')
out_corpus = PATHS['corpus'] / f'events_uschn_{DOWNLOAD_SCOPE}.parquet'
corpus.to_parquet(out_corpus, index=False)
print(f'Corpus: {len(corpus):,} US-CHN events -> {out_corpus}')
print('Keyword hit rate:', round(corpus['kw_text'].mean(), 4))


## 5 — Coverage diagnostic (catch empty decades early)


In [ ]:
cov = ml.coverage_table(corpus)
cov.to_csv(PATHS['validation'] / 'coverage_by_month.csv')
n_ok = int(cov['has_data'].sum())
print(f'Months with ≥1 event: {n_ok} / {len(ml.DATE_RANGE)}')
if n_ok:
    first = cov.index[cov['has_data']][0].strftime('%Y-%m')
    last = cov.index[cov['has_data']][-1].strftime('%Y-%m')
    print(f'  span: {first} -> {last}')
if n_ok < 300 and DOWNLOAD_SCOPE == 'saadaoui':
    print('WARNING: sparse coverage — consider saadaoui_dense or check caches.')

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.bar(cov.index, cov['n_events'], width=25, color='#8b0000', alpha=0.75)
ax.set_title('US-CHN events per month (raw GDELT, labeled)')
ax.set_ylabel('event count')
fig.tight_layout()
fig.savefig(PATHS['figures'] / 'coverage_events_per_month.png', dpi=150)
plt.show()


## 6 — Construct monthly GPR + Δ²

$$\text{gpr\_kw\_share}_t = \frac{\#\{\text{keyword-hit US–CHN events in } t\}}{\#\{\text{all US–CHN events in } t\}}$$

$$\Delta^2\text{gpr\_kw}_t = \text{gpr\_kw\_share}_t - 2\,\text{gpr\_kw\_share}_{t-1} + \text{gpr\_kw\_share}_{t-2}$$


In [ ]:
monthly = ml.monthly_gpr_from_events(corpus)
monthly['d2gpr_kw'] = ml.second_difference(monthly['gpr_kw_share'])
monthly['built_scope'] = DOWNLOAD_SCOPE
monthly['n_months_with_events'] = int(cov['has_data'].sum())

built_path = PATHS['constructed'] / 'gpr_uschn_built.csv'
monthly.to_csv(built_path)
print('Saved:', built_path)
print(monthly[['n_events_total','n_events_kw','gpr_kw_share','d2gpr_kw']].dropna(how='all').describe().round(4))

fig, ax = plt.subplots(figsize=(11, 3))
s = monthly['gpr_kw_share'].dropna()
ax.plot(s.index, s.values, color='#8b0000')
for yr in [1996, 1999, 2001, 2018, 2020]:
    ax.axvline(pd.Timestamp(f'{yr}-06-01'), color='gray', ls=':', alpha=0.4)
ax.set_title('Constructed gpr_kw_share'); ax.set_ylabel('share')
fig.savefig(PATHS['figures'] / 'gpr_kw_share_timeline.png', dpi=150)
plt.show()


## 7 — Validation (benchmarks only)

Strong correlation with Caldara GPR or Δ²PRI is **not required** for success — we test whether the built series moves with known tensions and Saadaoui’s PRI.


In [ ]:
df = pd.read_csv(PATHS['final'] / 'df_extended.csv', parse_dates=['Period_dt']).set_index('Period_dt')
df.index = df.index.to_period('M').to_timestamp()
m = df.join(monthly, how='left')

bench = PATHS['constructed'] / 'gpr_global_benchmark.csv'
if not bench.exists(): ml.download_caldara_gpr(bench)
m = m.join(pd.read_csv(bench, index_col=0, parse_dates=True), how='left')

rows = []
for col in ['gpr_kw_share','d2gpr_kw','gpr_global']:
    for tgt in ['lpri','d2pri','lwti','gpr_chn_l1']:
        if col not in m.columns or tgt not in m.columns: continue
        sub = m[[col,tgt]].dropna()
        if len(sub) < 20: continue
        r,p = stats.pearsonr(sub[col], sub[tgt])
        rows.append({'built':col,'vs':tgt,'r':round(r,3),'p':round(p,4),'n':len(sub)})
val = pd.DataFrame(rows)
val.to_csv(PATHS['validation'] / 'benchmark_correlations.csv', index=False)
print(val.to_string(index=False))


## 8 — First stage & IV-LP (built instrument vs Saadaoui)


In [ ]:
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# First-stage diagnostic: how well does d2gpr_kw predict lpri?
sub = m[['lpri','d2gpr_kw','L1_lpri','L2_lpri','vix']].dropna()
if len(sub) > 40:
    fs = sm.OLS(sub['lpri'],
                sm.add_constant(sub[['d2gpr_kw','L1_lpri','L2_lpri','vix']])).fit(
                cov_type='HAC', cov_kwds={'maxlags': 6})
    print('First-stage d2gpr_kw t-stat:', round(fs.tvalues.get('d2gpr_kw', float('nan')), 2))
    print('Compare: d2pri first-stage F ~ 178 (Saadaoui baseline)')
    print('Expect: weaker F for level-based d2gpr_kw vs turning-point d2pri')

# LP-IV: compare Saadaoui instrument vs your built d2gpr_kw
if RUN_IV:
    CTRL = [c for c in m.columns if c.startswith(('L1_', 'L2_')) or c in [
        'llwip','dllgop','dl2lgop','vix','gs10','brent','gold',
        'bdi','cny_usd','indpro','gpr_chn_l1','gpr_usa_l1']]
    res = []
    for instr, lab in [('d2pri', 'Saadaoui'), ('d2gpr_kw', 'Built')]:
        if instr not in m.columns:
            continue
        for h in range(H_MAX + 1):
            d = m.copy()
            d['_y'] = d['lwti'].shift(-h)
            cols = ['_y', 'lpri', instr] + CTRL
            s = d[[c for c in cols if c in d.columns]].dropna()
            if len(s) < 50:
                continue
            try:
                exog_iv = s[CTRL].copy()
                exog_iv.insert(0, 'const', 1.0)
                fit = IV2SLS(
                    s['_y'], exog_iv, s[['lpri']], s[[instr]]
                ).fit(cov_type='kernel')
                res.append({'h': h, 'instrument': lab, 'coef': float(fit.params['lpri'])})
            except Exception:
                pass
    irf = pd.DataFrame(res)
    irf.to_csv(PATHS['validation'] / 'irf_built_vs_d2pri.csv', index=False)
    if not irf.empty:
        print('LP-IV comparison (selected horizons):')
        pivot = irf.pivot(index='h', columns='instrument', values='coef')
        print(pivot.loc[[h for h in [0,3,6,12,24,36,48] if h in pivot.index]].round(4))

m.to_csv(PATHS['constructed'] / 'df_extended_with_built_gpr.csv')
print('Saved: df_extended_with_built_gpr.csv')


## 9 — Thesis paragraph & next steps

> We construct a bilateral US–China geopolitical risk index from raw GDELT 1.0 event records (yearly, monthly, and daily archives), defining keyword hits from threat-term lists applied to source URLs and actor fields. Caldara–Iacoviello GPR and Saadaoui’s Δ²PRI are used for validation only. Relative to Bondarenko et al. (2026), we proxy article-based GPR with event-linked news text; future work can add free Chinese corpora under `data/measurement/raw/chinese_news/`.

**For full Saadaoui window:** set `DOWNLOAD_SCOPE = 'saadaoui'` (then `'saadaoui_dense'` if professor wants every daily file 2013–2022). Clear bad cache: delete `data/measurement/raw/gdelt_daily/*` from the old daily-only pilot.
